# Naive Bayes for Sentiment Analysis

In this assignment you will implement and experiment with various feature engineering techniques in the context of Naive Bayes models for Sentiment classification of movie reviews.

We will use the NB model implemented in sklearn:

https://scikit-learn.org/stable/modules/naive_bayes.html

## Write Your Name Here: Basit Hussain

1.   List item
2.   List item



# <font color="blue"> Submission Instructions</font>

1. Click the Save button at the top of the Jupyter Notebook/Google Colab.
2. Please make sure to have entered your name above.
3. Select Cell -> All Output -> Clear. This will clear all the outputs from all cells (but will keep the content of all cells).
4. Select Cell -> Run All. This will run all the cells in order, and will take several minutes.
5. Once you've rerun everything, select File -> Download as (or Print) -> PDF and download a PDF version *SentimentAnalysis.pdf* showing the code and the output of all cells, and save it in the same folder that contains the notebook file *SentimentAnalysis.ipynb*.
6. Look at the PDF file and make sure all your solutions are there, displayed correctly. The PDF is the only thing we will see when grading!
7. Submit **both** your PDF and notebook on Canvas.
8. Verify your Canvas submission contains the correct files by downloading it after posting it on Canvas.

*Make sure that you format your solutions to theory questions to show equations properly. We will not grade solutions that are not properly formatted. Jupyter-notebook understands Latex, alternatively you can edit in Word using its Equation editor and submit the PDF as a separate file in Canvas.*

<hr>

*italicized text*# Theory (20p)

## Theory: J&M Exercise 4.1 (20p)
Your solution goes here ...

We are given the sentence:

“I always like foreign films.”
We are also given the following likelihoods and equal priors for both classes:

| Word | P(word | pos) | P(word | neg) | |---------|----------------|----------------| | I | 0.09 | 0.16 | | always | 0.07 | 0.06 | | like | 0.29 | 0.06 | | foreign | 0.04 | 0.15 | | films | 0.08 | 0.11 |

And: P (pos) = P (neg) = 0.5 P(pos)=P(neg)=0.5<hr>

We compute:

logP(pos | sentence)∝logP(pos)+ i ∑​ logP( Wi | pos)
logP(neg | sentence)∝logP(neg)+ i ∑​ logP(Wi | neg)

Positive class:
logP(pos)=log0.5= -0.6931
∑ logP(Wi | pos)=log(0.09)+log(0.07)+log(0.29)+log(0.04)+log(0.08)

Using natural logs (base e):
ln(0.09) ≈ -2.4079
ln(0.07) ≈ -2.6593
ln(0.29) ≈ -1.2379
ln(0.04) ≈ -3.2189
ln(0.08) ≈ -2.5257

So:

logP(pos | sentence)=- 0.6931 -2.4079 -2.6593 = -1.2379 -32189 -2.5257= 12.7428

Negative class:
logP(neg)=log0.5= -0.6931

∑ logP(Wi | neg)=log(0.16)+log(0.06)+log(0.06)+log(0.15)+log(0.11)

ln(0.16) ≈ -1.8326
ln(0.06) ≈ -2.8134
ln(0.06) ≈ -2.8134
ln(0.15) ≈ -1.8971
ln(0.11) ≈ -2.2073

So:

logP(neg | sentence)= -0.6931 -1.8326 -2.8134 -2.8134 -1.8971 -2.2073= -12.2569

Score for positive: -12.7428
Score for negative: -12.2569


# Implementation (80p + Bonus 30p)

## From documents to feature vectors
This section illustratess the prototypical components of machine learning pipeline for an NLP task, in this case document classification:

1. Read document examples (train, devel, test) from files with a predefined format:
    - assume one document per line, usign the format "\<label\> \<text\>".

2. Tokenize each document:
    - using a spaCy tokenizer.

3. Feature extractors:
    - so far, just words.

4. Process each document into a feature vector:
    - map document to a dictionary of feature names.
    - map feature names to unique feature IDs.
    - each document is a feature vector, where each feature ID is mapped to a feature value (e.g. word occurences).

In [1]:

import spacy
from spacy.lang.en import English
from scipy import sparse
from sklearn.naive_bayes import MultinomialNB

In [2]:
# Create spaCy tokenizer.
spacy_nlp = English()

def spacy_tokenizer(text):
    tokens = spacy_nlp.tokenizer(text)
    return [token.text for token in tokens]

In [3]:
# Read training and dev examples

def read_examples(filename):
    X = []
    Y = []
    with open(filename, mode='r', encoding='utf-8') as file:
        for line in file:
            [label, text] = line.rstrip().split(' ', maxsplit=1)
            X.append(text)
            Y.append(label)
    return X, Y


In [4]:
# Extract word-level features

def word_features(tokens):
    feats = {}
    for word in tokens:
        feat = 'WORD_%s' % word
        if feat in feats:
            feats[feat] += 1
        else:
            feats[feat] = 1
    return feats


In [5]:
# Merge feature dictionaries

def add_features(feats, new_feats):
    for feat in new_feats:
        if feat in feats:
            feats[feat] += new_feats[feat]
        else:
            feats[feat] = new_feats[feat]
    return feats


This function tokenizes the document, runs all the feature extractors on it and assembles the extracted features into a dictionary mapping feature names to feature values. It is important that feature names do not conflict with each other, i.e. different features should have different names. Each document will have its own dictionary of features and their values.

In [6]:
# This helper function converts a set of examples from a dictionary of feature names to values representation
# to a sparse representation of feature ids to values. This is important because almost all feature values will
# be 0 for most documents and it would be wasteful to save all in memory.

# Convert documents to features

def docs2features(trainX, feature_functions, tokenizer):
    examples = []
    count = 0
    for doc in trainX:
        feats = {}
        tokens = tokenizer(doc)
        for func in feature_functions:
            add_features(feats, func(tokens))
        examples.append(feats)
        count += 1
        if count % 100 == 0:
            print('Processed %d examples into features' % len(examples))
    return examples


In [7]:
# Assign feature names to IDs

def create_vocab(examples):
    feature_vocab = {}
    idx = 0
    for example in examples:
        for feat in example:
            if feat not in feature_vocab:
                feature_vocab[feat] = idx
                idx += 1
    return feature_vocab


In [8]:
# Convert to sparse matrix of feature ids

def features_to_ids(examples, feature_vocab):
    new_examples = sparse.lil_matrix((len(examples), len(feature_vocab)))
    for idx, example in enumerate(examples):
        for feat in example:
            if feat in feature_vocab:
                new_examples[idx, feature_vocab[feat]] = example[feat]
    return new_examples


In [9]:
# Evaluation pipeline for the Naive Bayes classifier.

def train_and_test(trainX, trainY, devX, devY, feature_functions, tokenizer):
    # Pre-process training documents.
    trainX_feat = docs2features(trainX, feature_functions, tokenizer)

    # Create vocabulary from features in training examples.
    feature_vocab = create_vocab(trainX_feat)
    print('Vocabulary size: %d' % len(feature_vocab))

    trainX_ids = features_to_ids(trainX_feat, feature_vocab)

    # Train NB model.
    nb_model = MultinomialNB(alpha = 1.0)
    nb_model.fit(trainX_ids, trainY)

    # Pre-process test documents.
    devX_feat = docs2features(devX, feature_functions, tokenizer)
    devX_ids = features_to_ids(devX_feat, feature_vocab)

    # Test NB model.
    print('Accuracy: %.3f' % nb_model.score(devX_ids, devY))


In [11]:
# For Colab only
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [12]:
# Set path and load files

import os

# For Colab
datapath = '/content/drive/MyDrive/data'

# For Notebook
# datapath = '../data'

train_file = os.path.join(datapath, 'imdb_sentiment_train.txt')
trainX, trainY = read_examples(train_file)

dev_file = os.path.join(datapath, 'imdb_sentiment_dev.txt')
devX, devY = read_examples(dev_file)

# Specify features to use.
features = [word_features]

# Evaluate NB model.
train_and_test(trainX, trainY, devX, devY, features, spacy_tokenizer)


Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Processed 1200 examples into features
Processed 1300 examples into features
Processed 1400 examples into features
Processed 1500 examples into features
Vocabulary size: 28692
Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Process

**bold text**## Feature engineering

[10p] Evaluate NB model performance when using only alpha tokens. This can be done by changing the tokenizer function.

In [13]:
def spacy_tokenizer1(text):
    tokens = spacy_nlp.tokenizer(text)

    # YOUR CODE HERE
    # Keep in the tokens list only those whose text is made up solely from letters.
    tokens = [token.text for token in tokens if token.text.isalpha()]

    return tokens



In [14]:
# Evaluate NB model.
train_and_test(trainX, trainY, devX, devY, features, spacy_tokenizer1)


Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Processed 1200 examples into features
Processed 1300 examples into features
Processed 1400 examples into features
Processed 1500 examples into features
Vocabulary size: 25054
Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Process

**bold text**[10p] Same as above, but lowercase all tokens before using as features.

In [15]:
def spacy_tokenizer2(text):
    tokens = spacy_nlp.tokenizer(text)

    # YOUR CODE HERE
    # Keep in the tokens list only those whose text is made up solely from letters.
    # Return a list of lowercased token text.
    tokens = [token.text.lower() for token in tokens if token.text.isalpha()]

    return tokens

# Evaluate NB model.
train_and_test(trainX, trainY, devX, devY, features, spacy_tokenizer2)


Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Processed 1200 examples into features
Processed 1300 examples into features
Processed 1400 examples into features
Processed 1500 examples into features
Vocabulary size: 21708
Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Process



```
# This is formatted as code
```

[15p] Same as above, but lowercase only tokens that appear at the beginning of sentences.

In [16]:
spacy_nlp = English()
spacy_nlp.add_pipe("sentencizer")

def spacy_tokenizer3(text):
    doc = spacy_nlp(text)

    tokens = []

    # YOUR CODE HERE
    for sent in doc.sents:
        for i, token in enumerate(sent):
            if token.text.isalpha():
                if i == 0:
                    tokens.append(token.text.lower())
                else:
                    tokens.append(token.text)

    return tokens

# Evaluate NB model.
train_and_test(trainX, trainY, devX, devY, features, spacy_tokenizer3)


Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Processed 1200 examples into features
Processed 1300 examples into features
Processed 1400 examples into features
Processed 1500 examples into features
Vocabulary size: 25054
Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Process

[15p] Use spacy_tokenizer2 (only alpha tokens, lowered all) and display the top 10 most frequent tokens in the vocabulary, as a list of tuples (token, frequency).

In [17]:
# First, count token occurrences across all examples, where features are still strings.
def create_feature_counts(examples):
    feature_counts = {}

    # YOUR CODE HERE
    for ex in examples:
        for feat, count in ex.items():
            if feat in feature_counts:
                feature_counts[feat] += count
            else:
                feature_counts[feat] = count

    return feature_counts

# Create features for all training examples, compute feature counts
def fcounts_from_train(trainX, feature_functions, tokenizer):
    # Pre-process training documents.
    trainX_feat = docs2features(trainX, feature_functions, tokenizer)

    # Create vocabulary from features in training examples.
    feature_counts = create_feature_counts(trainX_feat)
    print('Vocabulary size: %d' % len(feature_counts))

    return feature_counts

# Return a list of the top K most frequent tokens in the vocabulary.
def topK_tokens(vocab, k):
    # YOUR CODE HERE
    sorted_feats = sorted(vocab.items(), key=lambda x: x[1], reverse=True)
    return sorted_feats[:k]

# Run analysis with lowercase tokenizer
vocab = fcounts_from_train(trainX, features, spacy_tokenizer2)
stop_words = topK_tokens(vocab, 10)

# Display top 10 tokens
for item in stop_words:
    print(item)


Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Processed 1200 examples into features
Processed 1300 examples into features
Processed 1400 examples into features
Processed 1500 examples into features
Vocabulary size: 21708
('WORD_the', 20105)
('WORD_and', 9876)
('WORD_a', 9749)
('WORD_of', 9029)
('WORD_to', 8182)
('WORD_is', 6605)
('WORD_in', 5598)
('WORD_it', 5387)
('WORD_i', 4773)
('WORD_this', 4448)


[15p] Evaluate NB model performance when ignoring the top 20 stop words. Use spacy_tokenizer2 (only alpha tokens, lowered all).

In [18]:
# Evaluation pipeline for the Naive Bayes classifier.

def train_and_test(trainX, trainY, devX, devY, feature_functions, tokenizer):
    # Pre-process training documents.
    trainX_feat = docs2features(trainX, feature_functions, tokenizer)

    # Create vocabulary from features in training examples.
    feature_counts = create_feature_counts(trainX_feat)
    stop_words = topK_tokens(feature_counts, 20)

    # Remove from each example features that appear in the stop words list.
    # YOUR CODE HERE.
    for ex in trainX_feat:
        for sw, _ in stop_words:
            if sw in ex:
                del ex[sw]

    # Create vocabulary from features in training examples.
    feature_vocab = create_vocab(trainX_feat)
    print('Vocabulary size: %d' % len(feature_vocab))

    trainX_ids = features_to_ids(trainX_feat, feature_vocab)

    # Train NB model.
    nb_model = MultinomialNB(alpha = 1.0)
    nb_model.fit(trainX_ids, trainY)

    # Pre-process test documents.
    devX_feat = docs2features(devX, feature_functions, tokenizer)

    # Remove stop words from test data too
    for ex in devX_feat:
        for sw, _ in stop_words:
            if sw in ex:
                del ex[sw]

    devX_ids = features_to_ids(devX_feat, feature_vocab)

    # Test NB model.
    print('Accuracy: %.3f' % nb_model.score(devX_ids, devY))

# Evaluate NB model after stop word removal
train_and_test(trainX, trainY, devX, devY, features, spacy_tokenizer2)


Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Processed 1200 examples into features
Processed 1300 examples into features
Processed 1400 examples into features
Processed 1500 examples into features
Vocabulary size: 21688
Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Process

[5p] Evaluate NB model performance when ignoring words that appear less than 5 times. Use spacy_tokenizer2 (only alpha tokens, lowered all).

In [19]:
# Evaluation pipeline for the Naive Bayes classifier.

def train_and_test(trainX, trainY, devX, devY, feature_functions, tokenizer):
    # Pre-process training documents.
    trainX_feat = docs2features(trainX, feature_functions, tokenizer)

    # Create vocabulary from features in training examples.
    feature_counts = create_feature_counts(trainX_feat)

    # Remove features that appear less than 5 times.
    # YOUR CODE HERE.
    for ex in trainX_feat:
        for feat in list(ex.keys()):
            if feature_counts[feat] < 5:
                del ex[feat]

    # Create vocabulary from features in training examples.
    feature_vocab = create_vocab(trainX_feat)
    print('Vocabulary size: %d' % len(feature_vocab))

    trainX_ids = features_to_ids(trainX_feat, feature_vocab)

    # Train NB model.
    nb_model = MultinomialNB(alpha = 1.0)
    nb_model.fit(trainX_ids, trainY)

    # Pre-process test documents.
    devX_feat = docs2features(devX, feature_functions, tokenizer)

    # Remove low-frequency words not in vocab
    for ex in devX_feat:
        for feat in list(ex.keys()):
            if feat not in feature_vocab:
                del ex[feat]

    devX_ids = features_to_ids(devX_feat, feature_vocab)

    # Test NB model.
    print('Accuracy: %.3f' % nb_model.score(devX_ids, devY))

# Evaluate NB model with <5 frequency filter
train_and_test(trainX, trainY, devX, devY, features, spacy_tokenizer2)


Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Processed 1200 examples into features
Processed 1300 examples into features
Processed 1400 examples into features
Processed 1500 examples into features
Vocabulary size: 5573
Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Processe

## Analysis (10p)
Include an analysis of the results that you obtained in the experiments above. You ccan take advantage of the Jupyter Notebook markdown language, which can also process Latex and HTML, to format your report so that it looks professional.

Analysis :
We experimented with different preprocessing and feature engineering strategies to evaluate their impact on Naive Bayes performance in sentiment classification.


In this assignment, we evaluated how different feature engineering strategies affect the performance of a Naive Bayes classifier on sentiment analysis. Initially, using all tokens yielded a vocabulary size of 28,692 and an accuracy of 0.779. When we filtered out non-alphabetic tokens using spacy_tokenizer1, accuracy improved slightly to 0.785. Further lowercasing all tokens with spacy_tokenizer2 reduced the vocabulary to 21,708, with a stable accuracy of 0.784. Using spacy_tokenizer3, which only lowercases tokens at the beginning of sentences, yielded the same accuracy of 0.784 with a similar vocabulary size. The most significant improvement came from removing the top 20 most frequent (stop) words, increasing accuracy to 0.798. Finally, filtering out low-frequency words (those appearing fewer than 5 times) reduced the vocabulary dramatically to 5,573 and resulted in a slightly lower accuracy of 0.791. These results show that thoughtful token filtering and feature selection can both improve model performance and reduce computational complexity.

## [Bonus] Binary Multinomial Bayes (20p)
*Optional

Write code for transforming documents to features such that features are Boolean and only represent whether a word occurred in a document, as in the Binary Multinomial Naive Bayes discussed in class. Evaluate the Naive Bayes model with this feature representation, using spacy_tokenizer2 (only alpha tokens, lowered all).

In [20]:
%%writefile nbayes.py

import math
from collections import defaultdict

class NaiveBayes:
    def __init__(self):
        self.class_probs = {}
        self.feature_probs = {}
        self.vocab = set()
        self.classes = set()
        self.class_totals = defaultdict(int)

    def train(self, X, y):
        # Count features and class frequencies
        feature_counts = defaultdict(lambda: defaultdict(int))
        class_counts = defaultdict(int)

        for features, label in zip(X, y):
            class_counts[label] += 1
            self.class_totals[label] += sum(features.values())
            for feat, value in features.items():
                feature_counts[label][feat] += value
                self.vocab.add(feat)
            self.classes.add(label)

        # Calculate class priors
        total = len(y)
        self.class_probs = {c: math.log(class_counts[c] / total) for c in self.classes}

        # Calculate feature likelihoods with Laplace smoothing
        self.feature_probs = {}
        for c in self.classes:
            self.feature_probs[c] = {}
            for feat in self.vocab:
                count = feature_counts[c][feat]
                self.feature_probs[c][feat] = math.log((count + 1) / (self.class_totals[c] + len(self.vocab)))

    def predict(self, X):
        predictions = []
        for feats in X:
            scores = {}
            for c in self.classes:
                scores[c] = self.class_probs[c]
                for feat in feats:
                    if feat in self.feature_probs[c]:
                        scores[c] += feats[feat] * self.feature_probs[c][feat]
            predictions.append(max(scores, key=scores.get))
        return predictions


Writing nbayes.py


## Bonus points (30p)
*Optional
Implement NB from scratch in a separate module nbayes.py and use it for the exercises above.

In [21]:
from nbayes import NaiveBayes
from sklearn.metrics import accuracy_score

# Step 1: Create features
trainX_feat = docs2features(trainX, features, spacy_tokenizer2)
devX_feat = docs2features(devX, features, spacy_tokenizer2)

# Step 2: Train custom NB
nb = NaiveBayes()
nb.train(trainX_feat, trainY)

# Step 3: Predict
dev_preds = nb.predict(devX_feat)

# Step 4: Evaluate
acc = accuracy_score(devY, dev_preds)
print("Custom Naive Bayes Accuracy: %.3f" % acc)


Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Processed 1200 examples into features
Processed 1300 examples into features
Processed 1400 examples into features
Processed 1500 examples into features
Processed 100 examples into features
Processed 200 examples into features
Processed 300 examples into features
Processed 400 examples into features
Processed 500 examples into features
Processed 600 examples into features
Processed 700 examples into features
Processed 800 examples into features
Processed 900 examples into features
Processed 1000 examples into features
Processed 1100 examples into features
Processed 1200 examples into f